In [1]:
import pickle 
import pandas as pd
import numpy as np
import json

seeds = [0,1,2,3,42]
cancers = ['ccRCC', 'Melanoma', 'NSCLC']

## 1. Singling Out

In [2]:
def load_results_SO(path, is_uni = True):
    with open(path, 'rb') as file:
        linkabilityresults = pickle.load(file)
    
    tools = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    if is_uni:
        columns = ['25%', '50%', '75%', '100%']
    else:
        columns = [2,3,5,7,10,20,50]
    ResultasDataset = {}
    for tool, result_as_tool in linkabilityresults.items():
        tool_res = {}
        for i, evaluator in enumerate(result_as_tool):    
            attack_risk = evaluator.risk().value
            naive_risk = evaluator.risk(baseline=True).value
    
            if attack_risk <= naive_risk:
                final_risk = naive_risk
            else:
                final_risk = attack_risk
            tool_res[columns[i]] = final_risk
    
        ResultasDataset[tool] = tool_res
        
    df = pd.DataFrame(ResultasDataset).T
    df.index = tools
    df.columns = columns
    return df

In [3]:
## Uni Risk
UniRisk_Cancer = {}
for cancer in cancers:
    ResultsLinkability = {}
    for seed in seeds:
        data_path = f'../{cancer}/Privacy/SinglingOut/UniSO/Seed_{seed}/SOUni_Results.pkl'
        result_df = load_results_SO(data_path)
        mean_risk = result_df.mean(axis = 1)

        ResultsLinkability[seed] = mean_risk

    UniRisk_Cancer[cancer] = ResultsLinkability

/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/anonymeter/stats/confidence.py:218: UserWarning: Attack is as good or worse as baseline model. Estimated rates: attack = 0.0015914615876193507, baseline = 0.001691423187782283. Analysis results cannot be trusted.
  self._sanity_check()
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/anonymeter/stats/confidence.py:218: UserWarning: Attack is as good or worse as baseline model. Estimated rates: attack = 0.0007917687863158919, baseline = 0.0007917687863158919. Analysis results cannot be trusted.
  self._sanity_check()
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/anonymeter/stats/confidence.py:218: UserWarning: Attack is as good or worse as baseline model. Estimated rates: attack = 0.0009916919866417566, baseline = 0.007189311196743562. Analysis results cannot be trusted.
  self._sanity_check()
/binary/miniforge3/envs/synthetic_data/lib/python3.9/site-packages/anonymeter/stats/confid

In [ ]:
## Multi Risk
MultiRisk_Cancer = {}
for cancer in cancers:
    ResultsLinkability = {}
    for seed in seeds:
        data_path = f'../{cancer}/Privacy/SinglingOut/MultiSO/Seed_{seed}/MultiSO_Results.pkl'
        result_df = load_results_SO(data_path, is_uni = False)
        mean_risk = result_df.mean(axis = 1)

        ResultsLinkability[seed] = mean_risk

    MultiRisk_Cancer[cancer] = ResultsLinkability

## Linkability

In [ ]:
def load_results_linkability(path):
    with open(path, 'rb') as file:
        linkabilityresults = pickle.load(file)
    
    tools = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    columns = ['25%', '50%', '75%', '100%']
    ResultasDataset = {}
    for tool, result_as_tool in linkabilityresults.items():
        tool_res = {}
        for i, evaluator in enumerate(result_as_tool):
            # name_feature = feature[0]
            # evaluator = feature[1]
    
            attack_risk = evaluator.risk().value
            naive_risk = evaluator.risk(baseline=True).value
    
            if attack_risk <= naive_risk:
                final_risk = naive_risk
            else:
                final_risk = attack_risk
            tool_res[columns[i]] = final_risk
    
        ResultasDataset[tool] = tool_res
        
    df = pd.DataFrame(ResultasDataset).T
    df.index = tools
    df.columns = columns
    return df

In [ ]:
## Linkability Risk
LinkRisk_Cancer = {}
for cancer in cancers:
    ResultsLinkability = {}
    for seed in seeds:
        data_path = f'../{cancer}/Privacy/Linkability/Seed_{seed}/LinkabilityResults.pkl'
        result_df = load_results_linkability(data_path)
        mean_risk = result_df.mean(axis = 1)

        ResultsLinkability[seed] = mean_risk

    LinkRisk_Cancer[cancer] = ResultsLinkability

## Inference risk

In [ ]:
def load_results_inference(results): 
    
    with open(src_path, 'rb') as file:
        results_inference = pickle.load(file)
        
    ResultasDataset = {}
    tools = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    for tool, result_as_attributes in results_inference.items():
        tool_res = {}
        for feature in result_as_attributes:
            name_feature = feature[0]
            evaluator = feature[1]
    
            attack_risk = evaluator.risk().value
            naive_risk = evaluator.risk(baseline=True).value
    
            if attack_risk <= naive_risk:
                final_risk = naive_risk
            else:
                final_risk = attack_risk
            tool_res[name_feature] = final_risk
    
        ResultasDataset[tool] = tool_res
        
    df = pd.DataFrame(ResultasDataset).T
    df.index = tools

    return df

def extract_num_clincal(datapath, metadata, number_clinical):
    original_data = pd.read_csv(datapath, index_col = 0)
    clinical_features = original_data.columns.tolist()[0:number_clinical]
    numerical_type = ['numerical']
    numerical_features = [i.replace(".", "_") for i, value_type in metadata.items() if value_type in numerical_type and i in clinical_features]

    return numerical_features

def rearrange(data, numerical_clinical_features):
    df_numerical = data.loc[:,numerical_clinical_features]
    df_categorical = data.loc[:, ~df_results.columns.isin(numerical_clinical_features)]
    
    df_merged = pd.concat([df_numerical,df_categorical], axis = 1)

    return df_merged

In [ ]:
InferenceRisk_cancer = {}
for i, cancer in enumerate(cancers):
    Inference_as_seed = {}
    for seed in seeds:
        src_path = f'../{cancer}/Privacy/Inference/Seed_{seed}/results_inferences.pkl'
        # data_path = f'../{cancer}/Data/original_data.csv'
        df_results = load_results_inference(src_path)
        mean_risk = df_results.mean(axis = 1)
        Inference_as_seed[seed] = mean_risk
    InferenceRisk_cancer[cancer] = Inference_as_seed

## Score calculation

In [ ]:
OveralScore_dict = {}

for cancer in cancers:
    uni_df = pd.DataFrame(UniRisk_Cancer[cancer])
    multi_df = pd.DataFrame(MultiRisk_Cancer[cancer])
    link_df = pd.DataFrame(LinkRisk_Cancer[cancer])
    inference_df = pd.DataFrame(InferenceRisk_cancer[cancer])
    
    sdg_methods = ['Avatars K5', "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
    overalscore_dict = {}
    
    for i, tool in enumerate(sdg_methods):
        # Trích xuất các dòng dữ liệu dưới dạng array
        uni_risk = uni_df.iloc[i, :].values
        multi_risk = multi_df.iloc[i, :].values
        link_risk = link_df.iloc[i, :].values
        inference_risk = inference_df.iloc[i, :].values
        
        # Gom các mảng lại thành một ma trận (stack) để tính toán theo trục
        # Mỗi cột trong 'combined' sẽ chứa 4 giá trị rủi ro tương ứng
        combined = np.array([uni_risk, multi_risk, link_risk, inference_risk])
        
        # Sử dụng np.nanmean để tính trung bình, bỏ qua các giá trị np.nan
        # axis=0 giúp tính trung bình theo từng cột (từng seed)
        overal_risk = np.nanmean(combined, axis=0)
        
        # Tính Privacy Score: 1 - rủi ro
        overal_score = 1 - overal_risk
        
        overalscore_dict[tool] = overal_score
    
    # Tạo DataFrame kết quả
    # Lưu ý: 'seeds' cần phải trùng khớp với số lượng phần tử trong overal_score
    overal_score_df = pd.DataFrame(overalscore_dict).T
    overal_score_df.columns = seeds  # Gán tên cột là các seeds
    
    # Lưu file
    overal_score_df.to_csv(f'overal_privacy_score_{cancer}.csv', index=True)
    OveralScore_dict[cancer] = overal_score_df

In [ ]:
for cancer, overal_score_df in OveralScore_dict.items():
    print(f'-----{cancer}-----')
    stat_df = overal_score_df.T.describe()
    for tool in stat_df.columns.tolist():
        print(f"{tool}: {stat_df.loc['mean', tool]} +/- {stat_df.loc['std', tool]}")
        

In [ ]:
score_df = OveralScore_dict['ccRCC']
data = score_df.T.to_dict()
result = {key: list(inner_dict.values()) for key, inner_dict in data.items()}

## **Bayesian estimation**

In [ ]:
from SynOmics.metrics.narrow_utility.BayesianComparison import BayesianComparison
Heatmap_Dict = {}
for cancer, score_df in OveralScore_dict.items():
    data = score_df.T.to_dict()
    result = {key: list(inner_dict.values()) for key, inner_dict in data.items()}

    Heatmap_Dict[cancer] = result

In [ ]:
bc = BayesianComparison(rope=0.01)
methods_order = ["Avatars K5", "Avatars K10", "CTGAN", "Gaussian Copula", "Synthpop", "TVAE"]
fig, axes, mats_by_cancer, comps_by_cancer = bc.plot_pbetter_heatmap_grid(
    cancer_to_method_scores=Heatmap_Dict,
    cancers_order=("ccRCC", "Melanoma", "NSCLC"),
    methods_order=methods_order,
    value_col="Better Prob",
    figsize = (18,5),
    annot=True,
    fmt=".2f",
    show=True,
)

# fig.savefig("OverallPrivacy.pdf", bbox_inches="tight", facecolor="white")
for cancer in cancers:
    bayesian_comp_df = comps_by_cancer[cancer]
    # bayesian_comp_df.to_csv(f'OverallPrivacy_Bayesian_{cancer}.csv')